In [6]:
import os
from io import StringIO

import pandas as pd
from astropy.table import QTable


base_dir = "/home/dwich2006/binaries-research/NGC6866_6811"

clusters = {
    "NGC6811": {
        "catalog": os.path.join(base_dir, "rcat_ngc6811_v0.fits"),
        "tsv": os.path.join(base_dir, "NGC6811_nosingle.tsv"),
    },
    "NGC6866": {
        "catalog": os.path.join(base_dir, "rcat_ngc6866_v0.fits"),
        "tsv": os.path.join(base_dir, "NGC6866_nosingle.tsv"),
    },
}

output_file = os.path.join(base_dir, "All_NSS_Matches.csv")


def find_id_column(table):
    possible_columns = [
        "GAIAEDR3_ID",
        "GAIADR3_ID",
        "source_id",
        "SOURCE_ID",
        "Source",
        "source",
    ]

    for column in possible_columns:
        if column in table.colnames:
            return column

    raise KeyError(
        "Could not find the Gaia source-ID column.\n"
        f"Available columns: {table.colnames}"
    )


def read_cluster_ids(catalog_filename):
    catalog = QTable.read(catalog_filename)
    id_column = find_id_column(catalog)

    cluster_ids = set()

    for value in catalog[id_column]:
        value_string = str(value).strip()

        if value_string and value_string.lower() not in [
            "nan",
            "none",
            "--",
        ]:
            cluster_ids.add(value_string)

    print(f"Read {os.path.basename(catalog_filename)}")
    print(f"Gaia ID column: {id_column}")
    print(f"Number of Gaia IDs: {len(cluster_ids)}")

    return cluster_ids


def split_vizier_file(tsv_filename):
    with open(tsv_filename, "r", encoding="utf-8") as file:
        lines = file.readlines()

    table_start_indices = []

    for index, line in enumerate(lines):
        if line.startswith("#Table"):
            table_start_indices.append(index)

    print(
        f"Found {len(table_start_indices)} VizieR tables in "
        f"{os.path.basename(tsv_filename)}"
    )

    sections = []

    for index, start in enumerate(table_start_indices):
        if index + 1 < len(table_start_indices):
            end = table_start_indices[index + 1]
        else:
            end = len(lines)

        sections.append(lines[start:end])

    return sections


def get_table_information(section):
    table_name = ""
    catalog_name = ""
    title = ""

    for line in section:
        stripped = line.strip()

        if stripped.startswith("#Table"):
            table_name = stripped.replace("#Table", "", 1).strip()
            table_name = table_name.rstrip(":")

        elif stripped.startswith("#Name:"):
            catalog_name = stripped.replace("#Name:", "", 1).strip()

        elif stripped.startswith("#Title:"):
            title = stripped.replace("#Title:", "", 1).strip()

    return table_name, catalog_name, title


def parse_vizier_section(section):
    table_name, catalog_name, title = get_table_information(section)

    noncomment_lines = []

    for line in section:
        stripped = line.strip()

        if not stripped:
            continue

        if stripped.startswith("#"):
            continue

        noncomment_lines.append(line.rstrip("\n"))

    if len(noncomment_lines) < 4:
        return None, table_name, catalog_name, title

    header_line = noncomment_lines[0]
    data_lines = noncomment_lines[3:]

    if ";" not in header_line:
        return None, table_name, catalog_name, title

    if not data_lines:
        return None, table_name, catalog_name, title

    table_text = header_line + "\n" + "\n".join(data_lines)

    try:
        dataframe = pd.read_csv(
            StringIO(table_text),
            sep=";",
            dtype=str,
            keep_default_na=False,
            na_filter=False,
        )

    except Exception as error:
        print(f"Could not parse {table_name}")
        print(error)
        return None, table_name, catalog_name, title

    dataframe.columns = [
        str(column).strip()
        for column in dataframe.columns
    ]

    for column in dataframe.columns:
        dataframe[column] = dataframe[column].astype(str).str.strip()

    return dataframe, table_name, catalog_name, title


def find_source_column(dataframe):
    possible_columns = [
        "Source",
        "source",
        "source_id",
        "SOURCE_ID",
        "GAIAEDR3_ID",
        "GAIADR3_ID",
    ]

    for column in possible_columns:
        if column in dataframe.columns:
            return column

    return None


all_matches = []

for cluster_name, filenames in clusters.items():

    cluster_ids = read_cluster_ids(filenames["catalog"])
    sections = split_vizier_file(filenames["tsv"])

    for table_number, section in enumerate(sections, start=1):
        dataframe, table_name, catalog_name, title = parse_vizier_section(
            section
        )

        print("\n" + "-" * 100)
        print(f"Table {table_number}: {catalog_name}")

        if dataframe is None:
            print("No readable data rows.")
            continue

        source_column = find_source_column(dataframe)

        if source_column is None:
            print("No source-ID column found.")
            continue

        source_values = dataframe[source_column].astype(str).str.strip()

        matched = dataframe[source_values.isin(cluster_ids)].copy()

        print(f"Rows in table: {len(dataframe)}")
        print(f"Matched rows: {len(matched)}")

        if matched.empty:
            continue

        matched.insert(0, "Cluster", cluster_name)
        matched.insert(1, "NSS_Table_Number", table_number)
        matched.insert(2, "NSS_Table", catalog_name)
        matched.insert(3, "NSS_Title", title)

        if source_column != "Source":
            matched.rename(
                columns={source_column: "Source"},
                inplace=True,
            )

        all_matches.append(matched)


if all_matches:
    combined_matches = pd.concat(
        all_matches,
        ignore_index=True,
        sort=False,
    )

    combined_matches.to_csv(output_file, index=False)

    unique_stars = (
        combined_matches["Source"]
        .astype(str)
        .str.strip()
        .nunique()
    )

    print(f"Total matched rows: {len(combined_matches)}")
    print(f"Total unique matched stars: {unique_stars}")

    for cluster_name in clusters:
        cluster_matches = combined_matches[
            combined_matches["Cluster"] == cluster_name
        ]

        cluster_unique = (
            cluster_matches["Source"]
            .astype(str)
            .str.strip()
            .nunique()
        )

        print(
            f"{cluster_name}: "
            f"{len(cluster_matches)} matched rows, "
            f"{cluster_unique} unique stars"
        )

    print(f"\nSaved all matches to:\n{output_file}")

else:
    print("\nNo NSS matches were found.")

Read rcat_ngc6811_v0.fits
Gaia ID column: GAIAEDR3_ID
Number of Gaia IDs: 2716
Found 17 VizieR tables in NGC6811_nosingle.tsv

----------------------------------------------------------------------------------------------------
Table 1: I/357/tboasb1c
Rows in table: 1
Matched rows: 0

----------------------------------------------------------------------------------------------------
Table 2: I/357/tboeb
Rows in table: 5
Matched rows: 0

----------------------------------------------------------------------------------------------------
Table 3: I/357/tboes
No readable data rows.

----------------------------------------------------------------------------------------------------
Table 4: I/357/tbooc
Rows in table: 8
Matched rows: 4

----------------------------------------------------------------------------------------------------
Table 5: I/357/tbooac
No readable data rows.

----------------------------------------------------------------------------------------------------
Table 6:

FileNotFoundError: [Errno 2] No such file or directory: '/home/dwich2006/binaries-research/NGC6866_6811/NGC6866_nosingle.tsv'